## Configuration - set `bank` before running all cells

In [ ]:
bank = input("Enter bank name (e.g. 'asb' or 'anz'): ").strip().lower()
file_path = input("Enter path to labeled CSV file (e.g. 'data/ANZ-labelled-2026-04.csv'): ").strip()
print(f"Bank: {bank}")
print(f"Data file: {file_path}")

## Step 1 - Train a logistic regression classifier on labeled transactions

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sentence_transformers import SentenceTransformer

df = pd.read_csv(file_path)

# Drop rows with no category label
df = df[df['Category'].notna() & (df['Category'].str.strip() != '')]
print(f"Labeled rows: {len(df)}")
print(df['Category'].value_counts())

# Combine Payee, Memo and Tran Type for transaction context
df['transaction_context'] = (
    df['Payee'].fillna("") + " " +
    df['Memo'].fillna("") + " " +
    df['Tran Type'].fillna("")
).str.strip()

transaction_contexts = df['transaction_context'].tolist()
categories = df['Category'].tolist()

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(transaction_contexts, normalize_embeddings=True)

X_train, X_test, y_train, y_test = train_test_split(
    embeddings, categories, test_size=0.2, random_state=42
)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

joblib.dump(clf, f'models/{bank}_transaction_classifier.pkl')
joblib.dump(categories, f'models/{bank}_transaction_categories.pkl')

y_pred = clf.predict(X_test)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred, zero_division=1))

## Step 2 - Convert the trained model to ONNX format

In [ ]:
import joblib
import json
import skl2onnx
from skl2onnx.common.data_types import FloatTensorType

classifier = joblib.load(f'models/{bank}_transaction_classifier.pkl')
print("Model class:", classifier.__class__)
print("Number of features:", classifier.n_features_in_)

initial_type = [("float_input", FloatTensorType([None, classifier.n_features_in_]))]

onnx_model = skl2onnx.convert_sklearn(
    classifier,
    initial_types=initial_type,
    target_opset=13,
    options={'zipmap': False}
)
print("✅ ONNX model converted successfully!")

with open(f'models/{bank}_transaction_classifier.onnx', 'wb') as f:
    f.write(onnx_model.SerializeToString())
print(f"✅ ONNX model saved to models/{bank}_transaction_classifier.onnx")

categories = joblib.load(f'models/{bank}_transaction_categories.pkl')
with open(f'models/{bank}_transaction_categories.json', 'w') as f:
    json.dump(categories, f)
print(f"✅ Categories saved to models/{bank}_transaction_categories.json")

## Step 3 - Verify the ONNX model runs correctly

In [ ]:
import onnxruntime as ort
import numpy as np
import onnx
import json
from sentence_transformers import SentenceTransformer

model_path = f'models/{bank}_transaction_classifier.onnx'
print("🔍 Verifying ONNX Model:", model_path)
onnx_model = onnx.load(model_path)
print(f"✅ Model IR Version: {onnx_model.ir_version}")
print(f"✅ Opset Version: {onnx_model.opset_import[0].version}")

session = ort.InferenceSession(model_path)
print("✅ ONNX Model Loaded Successfully!\n")

with open(f'models/{bank}_transaction_categories.json') as f:
    categories = json.load(f)

embedder = SentenceTransformer('all-MiniLM-L6-v2')
input_name = session.get_inputs()[0].name

print("Enter payee names to test (one per line, empty line to finish):")
payees = []
while True:
    payee = input("  > ").strip()
    if not payee:
        break
    payees.append(payee)

if payees:
    embeddings = embedder.encode(payees, normalize_embeddings=True).astype(np.float32)
    outputs = session.run(None, {input_name: embeddings})
    print()
    for payee, category in zip(payees, outputs[0]):
        print(f"  {payee} → {category}")

In [ ]:
[KFC]